In [15]:
# ═══════════════════════════════════════════════════════════════════════
# CELL 0 — NAR Surgical Patch (Option B, PR #548)
# Patches the installed AR casanovo classes in-process at class level.
# Must run FIRST. No kernel restart needed — patches persist in memory,
# and this cell is now SAFE to re-run any number of times.
# ═══════════════════════════════════════════════════════════════════════
import subprocess, sys, warnings, inspect
warnings.filterwarnings('ignore')

subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
    'casanovo>=5.0.0', 'pyteomics', 'lxml', 'remotezip', 'appdirs'], check=True)

import torch
from casanovo.denovo.transformers import PeptideDecoder
from casanovo.denovo.model import Spec2Pep
from depthcharge.transformers import AnalyteTransformerDecoder  # parent class

# ─────────────────────────────────────────────────────────────────────
# FIX: capture the TRUE original embed() from the PARENT class
# (AnalyteTransformerDecoder), never from PeptideDecoder itself.
#
# Why this matters: in the AR (installed) package, PeptideDecoder does
# NOT define its own embed() — it inherits AnalyteTransformerDecoder's.
# The OLD version of this cell read `PeptideDecoder.embed`, which is
# fine on a fresh kernel, but if this cell is ever re-run in the SAME
# session, PeptideDecoder.embed is by then already our own _nar_embed —
# so the "original" captured on a second run would actually be the
# patch itself, and the patch would recursively call itself forever
# (RecursionError). Reading from AnalyteTransformerDecoder instead is
# immune to this: the patch only ever modifies PeptideDecoder, never
# its parent, so this reference is always the real, unpatched method —
# correct on the 1st run, the 2nd run, the 50th run, no exceptions.
# ─────────────────────────────────────────────────────────────────────
_ar_embed_original = AnalyteTransformerDecoder.embed

# ─────────────────────────────────────────────────────────────────────
# PATCH 1 — PeptideDecoder.embed
# Source: PR #548 transformers.py
# Replaces the causal upper-triangular tgt_mask the parent builds with
# an all-False (L×L) mask, allowing every position to attend to every
# other position — the core of non-autoregressive parallel decoding.
# `_orig` is bound as a DEFAULT ARGUMENT (evaluated once, at definition
# time) — a second, independent layer of protection against the
# late-binding recursion bug, even if `_ar_embed_original` were ever
# reassigned elsewhere.
# ─────────────────────────────────────────────────────────────────────
def _nar_embed(self, tokens, *args,
               memory,
               memory_key_padding_mask=None,
               memory_mask=None,
               tgt_mask=None,
               _orig=_ar_embed_original,
               **kwargs):
    if tokens is None:
        tokens = torch.tensor([[]], dtype=torch.float,
                               device=next(self.parameters()).device)
    L = tokens.shape[1] + 1          # +1 for the prepended global token
    tgt_mask = torch.zeros((L, L), dtype=torch.bool, device=tokens.device)
    return _orig(
        self, tokens, *args,
        memory=memory,
        memory_key_padding_mask=memory_key_padding_mask,
        memory_mask=memory_mask,
        tgt_mask=tgt_mask,
        **kwargs,
    )

# Fail fast, here, with a clear message — instead of a 3000-frame
# RecursionError surfacing minutes later inside a timing loop.
assert _ar_embed_original is not _nar_embed, (
    'BUG: captured "original" embed() is the NAR patch itself — '
    'this would cause infinite recursion. Restart the kernel and re-run.'
)

PeptideDecoder.embed = _nar_embed

# ─────────────────────────────────────────────────────────────────────
# PATCH 2 — Spec2Pep._forward_step
# Source: PR #548 model.py
# AR version feeds ground-truth tokens (teacher forcing).
# NAR version feeds an all-zero tensor so the decoder predicts ALL
# positions in a single parallel pass, with no access to prior outputs.
# This patch fully REPLACES the method body (no delegation to a captured
# "original"), so unlike Patch 1 it is inherently safe to re-run any
# number of times — there is nothing here that can self-recurse.
# ─────────────────────────────────────────────────────────────────────
def _nar_forward_step(self, batch):
    mzs, ints, precursors, seqs = self._process_batch(batch)
    dev = self.device
    mzs        = mzs.to(dev)
    ints       = ints.to(dev)
    precursors = precursors.to(dev)

    memories, mem_masks = self.encoder(mzs, ints)

    if seqs is not None:                    # training: match GT length
        zero_tokens = torch.zeros_like(seqs.to(dev))
    else:                                   # inference: full max length
        zero_tokens = torch.zeros(
            (mzs.shape[0], self.max_peptide_len),
            dtype=torch.long, device=dev,
        )
    scores = self.decoder(
        tokens=zero_tokens,
        memory=memories,
        memory_key_padding_mask=mem_masks,
        precursors=precursors,
    )
    return scores, seqs

Spec2Pep._forward_step = _nar_forward_step

# ─────────────────────────────────────────────────────────────────────
# PATCH 3 — Spec2Pep.forward
# Source: PR #548 model.py
# AR version called beam_search_decode; NAR delegates to _forward_step.
# Also a full replacement — re-run-safe for the same reason as Patch 2.
# ─────────────────────────────────────────────────────────────────────
def _nar_forward(self, batch):
    return self._forward_step(batch)

Spec2Pep.forward = _nar_forward

# ── Verification ──────────────────────────────────────────────────────
try:
    _ok_embed = 'tgt_mask = torch.zeros' in inspect.getsource(_nar_embed)
    _ok_fwd   = 'zero_tokens' in inspect.getsource(_nar_forward_step)
except (OSError, TypeError):
    _ok_embed = True
    _ok_fwd   = True

_ok_id_embed = (PeptideDecoder.embed  is _nar_embed)
_ok_id_fwd   = (Spec2Pep._forward_step is _nar_forward_step)
_ok_id_fwd2  = (Spec2Pep.forward       is _nar_forward)

print('\n── NAR Patch Status ──────────────────────────────────────────')
print(f'  PeptideDecoder.embed  patched (identity)  : {"✓" if _ok_id_embed else "✗ FAILED"}')
print(f'  Spec2Pep._forward_step patched (identity) : {"✓" if _ok_id_fwd   else "✗ FAILED"}')
print(f'  Spec2Pep.forward       patched (identity) : {"✓" if _ok_id_fwd2  else "✗ FAILED"}')
print(f'  all-False tgt_mask in embed source        : {"✓" if _ok_embed    else "~ (source check skipped)"}')
print(f'  zero_tokens in _forward_step source        : {"✓" if _ok_fwd      else "~ (source check skipped)"}')
print(f'  embed() original sourced from parent class : ✓ (recursion-safe, re-run-safe)')
print('──────────────────────────────────────────────────────────────')

if not (_ok_id_embed and _ok_id_fwd and _ok_id_fwd2):
    raise RuntimeError('One or more NAR patches failed — check errors above.')

print('\nNAR patches applied ✓  Safe to re-run this cell at any time.')
print('Continue → run Cell 1 (Setup) next.')


── NAR Patch Status ──────────────────────────────────────────
  PeptideDecoder.embed  patched (identity)  : ✓
  Spec2Pep._forward_step patched (identity) : ✓
  Spec2Pep.forward       patched (identity) : ✓
  all-False tgt_mask in embed source        : ✓
  zero_tokens in _forward_step source        : ✓
  embed() original sourced from parent class : ✓ (recursion-safe, re-run-safe)
──────────────────────────────────────────────────────────────

NAR patches applied ✓  Safe to re-run this cell at any time.
Continue → run Cell 1 (Setup) next.



[notice] A new release of pip is available: 25.1.1 -> 26.1.2
[notice] To update, run: pip install --upgrade pip


In [16]:
# ═══════════════════════════════════════════════════════════════════════
# CELL 1 — Setup
# ═══════════════════════════════════════════════════════════════════════
import subprocess, sys, os, time, warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import datetime
import matplotlib; matplotlib.use('Agg')
import matplotlib.pyplot as plt
import torch
from torch.profiler import profile, ProfilerActivity, schedule, record_function
from pathlib import Path
from tqdm import tqdm

torch.manual_seed(42)

DEVICE     = 'cuda' if torch.cuda.is_available() else 'cpu'
GPU_NAME   = torch.cuda.get_device_name(0) if DEVICE == 'cuda' else 'CPU'
TOTAL_VRAM = torch.cuda.get_device_properties(0).total_memory / 1e9 if DEVICE == 'cuda' else 0

N_PEAKS = 150     # only used to size the synthetic micro-benchmark spectra
                  # (Cell 9) — no longer needed for real-spectra timing since
                  # CUDA Graphs (which required fixed shapes) are removed.
AVG_PEP = 12

# ── NAR profiling constants ─────────────────────────────────────────
N_SUBSET         = 6000
N_TIMING_SPECTRA = 5000
BATCH_SIZES      = [1, 8, 32, 128, 512]
N_WARMUP_BATCHES = 10

PROF_WARMUP = 20
PROF_ACTIVE = 50          # ≥50 real spectra profiled in detail

# ── BF16 mixed-precision settings ─────────────────────────────────────
# Official inference pattern (PyTorch docs, torch.amp):
#   with torch.no_grad(), torch.autocast(device_type='cuda', dtype=torch.bfloat16):
#       output = model(input)
# Model weights are NEVER cast (no model.to(bfloat16)/.half()) — only the
# forward-pass compute runs in BF16; autocast handles op-by-op dtype choice.
BF16_DTYPE     = torch.bfloat16
BF16_SUPPORTED = torch.cuda.is_bf16_supported() if DEVICE == 'cuda' else False

def _sync():
    if DEVICE == 'cuda':
        torch.cuda.synchronize()

print(f'Device : {DEVICE} | GPU: {GPU_NAME} | VRAM: {TOTAL_VRAM:.1f} GB')
print(f'PyTorch: {torch.__version__} | CUDA: {torch.version.cuda}')
print(f'N_SUBSET={N_SUBSET} | N_TIMING_SPECTRA={N_TIMING_SPECTRA}')
print(f'Batch sizes : {BATCH_SIZES}')
print(f'BF16 supported on this GPU: {BF16_SUPPORTED}')
if DEVICE == 'cuda' and not BF16_SUPPORTED:
    print('  WARNING: this GPU does not support bfloat16 — BF16 cells will fail. '
          'Consider torch.float16 instead (requires GradScaler-style care for over/underflow).')
print(f'Profiler    : warmup={PROF_WARMUP} active={PROF_ACTIVE} (bs=1 only)')
os.makedirs('results', exist_ok=True)

Device : cuda | GPU: NVIDIA L4 | VRAM: 23.6 GB
PyTorch: 2.7.1+cu128 | CUDA: 12.8
N_SUBSET=6000 | N_TIMING_SPECTRA=5000
Batch sizes : [1, 8, 32, 128, 512]
BF16 supported on this GPU: True
Profiler    : warmup=20 active=50 (bs=1 only)


In [17]:
# ═══════════════════════════════════════════════════════════════════
# CELL 2 — Download MGF via HTTP range request (no full zip)
# ═══════════════════════════════════════════════════════════════════
from remotezip import RemoteZip
import shutil
 
ZIP_URL  = 'https://zenodo.org/records/12587317/files/mgf_data.zip?download=1'
TARGET   = 'multi-enzyme-simple.test.mgf'
MGF_PATH = TARGET
 
if not (os.path.exists(MGF_PATH) and os.path.getsize(MGF_PATH) > 1e6):
    with RemoteZip(ZIP_URL) as zf:
        src = next(n for n in zf.namelist() if TARGET in n)
        zf.extract(src, '.')
    if src != MGF_PATH and os.path.exists(src):
        shutil.move(src, MGF_PATH)
        top = src.split('/')[0]
        if os.path.isdir(top): shutil.rmtree(top, ignore_errors=True)
 
print(f'{MGF_PATH}  ({os.path.getsize(MGF_PATH)/1e6:.1f} MB)')

multi-enzyme-simple.test.mgf  (300.9 MB)


In [18]:
# ═══════════════════════════════════════════════════════════════════
# CELL 3 — EDA: parse MGF + plots
# ═══════════════════════════════════════════════════════════════════
import re as _re
def parse_mgf(path):
    records, spec, peaks, in_s = [], {}, [], False
    with open(path, 'r', errors='replace') as f:
        for line in f:
            line = line.strip()
            if not line: continue
            if line.upper() == 'BEGIN IONS':
                spec, peaks, in_s = {}, [], True
            elif line.upper() == 'END IONS':
                if in_s:
                    records.append({'pepmass': spec.get('_pm', 0.0),
                                    'charge':  spec.get('_ch', 1),
                                    'n_peaks': len(peaks)})
                in_s = False
            elif in_s:
                if '=' in line:
                    k, _, v = line.partition('='); k = k.strip().upper()
                    if k == 'PEPMASS': spec['_pm'] = float(v.strip().split()[0])
                    elif k == 'CHARGE': spec['_ch'] = int(_re.sub(r'[^\d]', '', v.strip()) or '1')
                else:
                    p = line.split()
                    if p:
                        try: peaks.append(float(p[0]))
                        except ValueError: pass
    return pd.DataFrame(records)
 
eda = parse_mgf(MGF_PATH)
print(f'Spectra : {len(eda):,}')
print(f'Charge  : +{eda.charge.min()} to +{eda.charge.max()} | '
      f'+2: {int((eda.charge==2).sum()):,}  +3: {int((eda.charge==3).sum()):,}')
print(f'm/z     : {eda.pepmass.min():.1f} – {eda.pepmass.max():.1f}')
print(f'Peaks   : {eda.n_peaks.mean():.0f} avg  (min {eda.n_peaks.min()}, max {eda.n_peaks.max()})')
 
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
fig.suptitle('EDA — multi-enzyme-simple.test.mgf', fontweight='bold')
vc = eda.charge.value_counts().sort_index()
axes[0].bar(vc.index.astype(str), vc.values, color='steelblue', edgecolor='white')
axes[0].set(title='Charge distribution', xlabel='Charge', ylabel='Count')
for bar, v in zip(axes[0].patches, vc.values):
    axes[0].text(bar.get_x()+bar.get_width()/2, bar.get_height()+max(vc.values)*0.01,
                 f'{v:,}', ha='center', fontsize=7)
axes[1].hist(eda.pepmass.clip(upper=3000), bins=60, color='darkorange', edgecolor='white')
axes[1].set(title='Precursor m/z (clip @3000)', xlabel='m/z', ylabel='Count')
axes[2].hist(eda.n_peaks.clip(upper=500), bins=60, color='seagreen', edgecolor='white')
axes[2].set(title='Peaks/spectrum (clip @500)', xlabel='Peaks', ylabel='Count')
plt.tight_layout()
plt.savefig('results/eda_plots.png', dpi=150, bbox_inches='tight'); plt.show()
print('Saved: results/eda_plots.png')

Spectra : 106,933
Charge  : +1 to +8 | +2: 40,614  +3: 39,146
m/z     : 301.2 – 1604.3
Peaks   : 123 avg  (min 6, max 950)
Saved: results/eda_plots.png


In [19]:
 
# ═══════════════════════════════════════════════════════════════════
# CELL 4 — Loading Casanovo model via Python API
# Uses the exact same checkpoint-finding function the CLI uses.
# ModelRunner handles all Lightning version compatibility internally.
# ═══════════════════════════════════════════════════════════════════
import appdirs
from casanovo.casanovo import _get_model_weights
from casanovo.denovo.model_runner import ModelRunner
from casanovo.config import Config
 
cache_dir = Path(appdirs.user_cache_dir("casanovo", False, opinion=False))
print(f'Cache dir : {cache_dir}')
ckpt_path  = _get_model_weights(cache_dir)   # finds cached or downloads
print(f'Checkpoint: {ckpt_path}  ({os.path.getsize(str(ckpt_path))/1e6:.0f} MB)')
 
config = Config()
runner = ModelRunner(config=config, model_filename=str(ckpt_path))
runner.initialize_tokenizer()
runner.initialize_model(train=False)
model  = runner.model.to(DEVICE).eval()
 
n_params = sum(p.numel() for p in model.parameters())
print(f'\nModel class : {type(model).__name__}')
print(f'Parameters  : {n_params/1e6:.1f}M')
print(f'dim_model   : {model.encoder.latent_spectrum.shape[-1]}')
print(f'Enc layers  : {model.encoder.transformer_encoder.num_layers}')
try:
    print(f'Dec layers  : {model.decoder.transformer_decoder.num_layers}')
except AttributeError:
    pass
print(f'n_beams     : {model.n_beams} | max_peptide_len: {model.max_peptide_len}')
# ── Verify NAR patches are live on the loaded model ─────────────────
print('\n── NAR verification on loaded model ──')
import inspect as _insp
try:
    _v_embed = 'tgt_mask = torch.zeros' in _insp.getsource(model.decoder.__class__.embed)
    _v_fwd   = 'zero_tokens'            in _insp.getsource(model._forward_step)
except (OSError, TypeError):
    _v_embed = (model.decoder.__class__.embed    is _nar_embed)
    _v_fwd   = (model.__class__._forward_step    is _nar_forward_step)

print(f'  Decoder embed → full attention (NAR) : {"✓" if _v_embed else "✗  — re-run Cell 0!"}')
print(f'  _forward_step → zero tokens (NAR)    : {"✓" if _v_fwd   else "✗  — re-run Cell 0!"}')
print(f'  beam_search_decode present (unused)  : {hasattr(model, "beam_search_decode")}')
if not (_v_embed and _v_fwd):
    raise RuntimeError('NAR patches not active on loaded model. Run Cell 0 first.')
print(f'  max_peptide_len = {model.max_peptide_len}')

Checkpoint directory not set in ModelRunner, no checkpoint files will be saved.
Configured residue(s) not in model alphabet: [Ammonia-loss]-, Q[Deamidated], [Acetyl]-, N[Deamidated], [+25.980265]-, [Carbamyl]-, M[Oxidation], C[Carbamidomethyl]


Cache dir : /home/zeus/.cache/casanovo
Checkpoint: /home/zeus/.cache/casanovo/casanovo_v5_0_0_v5_0_0.ckpt  (575 MB)

Model class : Spec2Pep
Parameters  : 47.9M
dim_model   : 512
Enc layers  : 9
Dec layers  : 9
n_beams     : 1 | max_peptide_len: 100

── NAR verification on loaded model ──
  Decoder embed → full attention (NAR) : ✓
  _forward_step → zero tokens (NAR)    : ✓
  beam_search_decode present (unused)  : True
  max_peptide_len = 100


In [20]:
# # ═══════════════════════════════════════════════════════════════════════
# # CELL 4b (NEW) — Compile encoder + decoder with torch.compile (CUDA Graphs)
# # Insert as a NEW cell immediately after Cell 4 (model loading).
# #
# # These objects are created ONCE here and reused everywhere below
# # (Cells 7, 8, 9) — creating a new torch.compile() wrapper discards any
# # previously cached compilation, so re-creating it per cell would force
# # an expensive recompile every time.
# # ═══════════════════════════════════════════════════════════════════════
# compiled_encoder = torch.compile(model.encoder, mode=COMPILE_MODE)
# compiled_decoder = torch.compile(model.decoder, mode=COMPILE_MODE)

# print(f'compiled_encoder / compiled_decoder created  (mode={COMPILE_MODE})')
# print('Note: the FIRST call at each new (module, shape) combination triggers')
# print('a one-time TorchInductor compile + CUDA Graph capture (10-60+ seconds).')
# print('This happens automatically inside the warm-up loop of Cell 7 / 8 / 9')
# print('below and is excluded from all timed results.')

In [21]:
# ═══════════════════════════════════════════════════════════════════════
# CELL 5 — Build 6 000-spectrum MGF subset + Lance DataModule
# Auto-rebuilds the Lance if N_SUBSET or max_charge changed from the
# previous run (e.g. old subset had only 100 spectra).
# ═══════════════════════════════════════════════════════════════════════
from casanovo.denovo.dataloaders import DeNovoDataModule
import shutil as _shutil

SUBSET_MGF = 'subset_profile.mgf'
LANCE_DIR  = os.path.join(os.getcwd(), 'lance_cache')
os.makedirs(LANCE_DIR, exist_ok=True)

MODEL_MAX_CHARGE = model.decoder.charge_encoder.num_embeddings
print(f'Model max_charge: {MODEL_MAX_CHARGE}')

# ── Helpers ──────────────────────────────────────────────────────────
def _count_mgf_spectra(path):
    if not os.path.exists(path):
        return 0
    with open(path, 'r', errors='replace') as f:
        return f.read().count('BEGIN IONS')

def write_subset_mgf(src, dest, n):
    count, buf, in_s = 0, [], False
    with open(src, 'r', errors='replace') as fin, open(dest, 'w') as fout:
        for line in fin:
            if count >= n:
                break
            if line.strip().upper() == 'BEGIN IONS':
                in_s = True; buf = [line]
            elif line.strip().upper() == 'END IONS':
                buf.append(line); fout.writelines(buf)
                count += 1; in_s = False; buf = []
            elif in_s:
                buf.append(line)
    return count

# ── Rebuild MGF subset if it is missing or too small ────────────────
_existing_n = _count_mgf_spectra(SUBSET_MGF)
if _existing_n < N_SUBSET:
    if _existing_n > 0:
        print(f'Old SUBSET_MGF has only {_existing_n} spectra (need {N_SUBSET}). Rebuilding…')
        os.remove(SUBSET_MGF)
    wrote = write_subset_mgf(MGF_PATH, SUBSET_MGF, N_SUBSET)
    print(f'Created: {SUBSET_MGF}  ({wrote} spectra)')
else:
    print(f'Reusing: {SUBSET_MGF}  ({_existing_n} spectra)')

# ── Rebuild Lance if N_SUBSET or max_charge changed ──────────────────
# Cache key encodes both so any change triggers a clean rebuild.
_mc_marker  = os.path.join(LANCE_DIR, '.cache_key')
_lance_test = os.path.join(LANCE_DIR, 'test.lance')
_cache_key  = f'{MODEL_MAX_CHARGE}_{N_SUBSET}'
_prev_key   = open(_mc_marker).read().strip() if os.path.exists(_mc_marker) else 'none'

if _prev_key != _cache_key:
    if os.path.exists(_lance_test):
        _shutil.rmtree(_lance_test)
        print(f'Deleted stale Lance (was {_prev_key!r}, now {_cache_key!r})')
    with open(_mc_marker, 'w') as f:
        f.write(_cache_key)
    print(f'Building Lance  cache_key={_cache_key} …')
else:
    print(f'Reusing Lance   cache_key={_cache_key} ✓')

# ── DataModule (bs=1 for setup / sanity check) ───────────────────────
dm = DeNovoDataModule(
    lance_dir=LANCE_DIR,
    test_paths=[SUBSET_MGF],
    eval_batch_size=1,
    tokenizer=runner.model.tokenizer,
    max_charge=MODEL_MAX_CHARGE,
    n_workers=0,
)
dm.setup(stage='test', annotated=False)
print('DataModule (bs=1) ready.')

# ── Sanity-check first batch ─────────────────────────────────────────
_it   = iter(dm.predict_dataloader())
_b    = next(_it); del _it
_mzs, _ints, _precs, _ = model._process_batch(_b)
_charge = _precs[0, 1].item() if _precs.ndim == 2 else _precs[1].item()
assert _charge <= MODEL_MAX_CHARGE, \
    f'Charge {_charge} > model max_charge {MODEL_MAX_CHARGE}'
print(f'First batch  mzs={_mzs.shape}  precs={_precs.shape}')
print(f'Precursor [0]: mass={_precs[0,0]:.1f}  charge={_charge:.0f}  mz={_precs[0,2]:.1f} ✓')
print(f'Subset ready: {N_SUBSET} spectra  (timing target: {N_TIMING_SPECTRA})')

Model max_charge: 4
Reusing: subset_profile.mgf  (6000 spectra)
Reusing Lance   cache_key=4_6000 ✓


subset_profile.mgf: 0 spectra [00:00, ? spectra/s]

DataModule (bs=1) ready.
First batch  mzs=torch.Size([1, 42])  precs=torch.Size([1, 3])
Precursor [0]: mass=3370.5  charge=3  mz=1124.5 ✓
Subset ready: 6000 spectra  (timing target: 5000)


In [22]:
# ═══════════════════════════════════════════════════════════════════════
# CELL 6 — Baseline NAR Timing (FP32, eager mode)
# Reverted to natural DataLoader batching — the fixed N_PEAKS padding
# from the torch.compile experiment is no longer needed (that was only
# required for CUDA Graphs' static-shape constraint, which is removed).
# ═══════════════════════════════════════════════════════════════════════
import threading

_gpu_samples_bl = []
_stop_gpu_bl    = threading.Event()

def _gpu_monitor_bl():
    import subprocess as _sp
    while not _stop_gpu_bl.is_set():
        r = _sp.run(['nvidia-smi',
                     '--query-gpu=utilization.gpu,memory.used',
                     '--format=csv,noheader,nounits'],
                    capture_output=True, text=True)
        if r.returncode == 0:
            try:
                u, m = r.stdout.strip().split(', ')
                _gpu_samples_bl.append((int(u), float(m) / 1024))
            except Exception:
                pass
        time.sleep(0.5)

threading.Thread(target=_gpu_monitor_bl, daemon=True).start()

timing_baseline = {}

for bs in BATCH_SIZES:
    print(f'\n══ Timing BASELINE NAR (FP32, eager)  batch_size={bs:4d} ══')

    _dm_bs = DeNovoDataModule(
        lance_dir=LANCE_DIR,
        test_paths=[SUBSET_MGF],
        eval_batch_size=bs,
        tokenizer=runner.model.tokenizer,
        max_charge=MODEL_MAX_CHARGE,
        n_workers=0,
    )
    _dm_bs.setup(stage='test', annotated=False)

    _w_iter = iter(_dm_bs.predict_dataloader())
    with torch.no_grad():
        for _w in range(N_WARMUP_BATCHES):
            try:
                _wb = next(_w_iter)
            except StopIteration:
                break
            _wm, _wi, _wp, _ = model._process_batch(_wb)
            _wm = _wm.to(DEVICE); _wi = _wi.to(DEVICE); _wp = _wp.to(DEVICE)
            _wme, _wmk = model.encoder(_wm, _wi)
            _wz = torch.zeros((_wm.shape[0], model.max_peptide_len),
                               dtype=torch.long, device=DEVICE)
            model.decoder(tokens=_wz, memory=_wme,
                          memory_key_padding_mask=_wmk, precursors=_wp)
    del _w_iter
    _sync()

    _t = {k: [] for k in ['fetch', 'h2d', 'enc', 'nar', 'write', 'total', 'tp']}
    _loader = _dm_bs.predict_dataloader()
    _it     = iter(_loader)
    n_spec  = 0
    pbar    = tqdm(total=N_TIMING_SPECTRA, desc=f'  bs={bs}', unit='spec')

    while n_spec < N_TIMING_SPECTRA:
        _sync(); t0 = time.perf_counter()
        try:
            batch = next(_it)
        except StopIteration:
            _it = iter(_loader)
            batch = next(_it)
        t_fetch = (time.perf_counter() - t0) * 1000

        _sync(); t0 = time.perf_counter()
        mzs, ints, precs, _ = model._process_batch(batch)
        mzs   = mzs.to(DEVICE)
        ints  = ints.to(DEVICE)
        precs = precs.to(DEVICE)
        _sync(); t_h2d = (time.perf_counter() - t0) * 1000
        actual_bs = mzs.shape[0]

        with torch.no_grad():
            _sync(); t0 = time.perf_counter()
            memories, mem_masks = model.encoder(mzs, ints)
            _sync(); t_enc = (time.perf_counter() - t0) * 1000

            zero_tokens = torch.zeros(
                (actual_bs, model.max_peptide_len),
                dtype=torch.long, device=DEVICE)
            _sync(); t0 = time.perf_counter()
            scores = model.decoder(
                tokens=zero_tokens,
                memory=memories,
                memory_key_padding_mask=mem_masks,
                precursors=precs,
            )
            _sync(); t_nar = (time.perf_counter() - t0) * 1000

        t0 = time.perf_counter()
        pred_tok = scores.argmax(dim=-1).cpu()
        _out = [{'tokens': tok.tolist()} for tok in pred_tok]
        t_write = (time.perf_counter() - t0) * 1000

        t_total = t_fetch + t_h2d + t_enc + t_nar + t_write

        _t['fetch'].append(t_fetch  / actual_bs)
        _t['h2d'].append(t_h2d     / actual_bs)
        _t['enc'].append(t_enc     / actual_bs)
        _t['nar'].append(t_nar     / actual_bs)
        _t['write'].append(t_write / actual_bs)
        _t['total'].append(t_total / actual_bs)
        _t['tp'].append(actual_bs  / (t_total / 1000))

        n_spec += actual_bs
        pbar.update(actual_bs)
        if n_spec >= N_TIMING_SPECTRA:
            break

    pbar.close()

    def _p(a, q): return np.percentile(a, q)
    timing_baseline[bs] = {
        'n_spec'     : n_spec,
        'n_batches'  : len(_t['total']),
        'fetch_mean' : np.mean(_t['fetch']),
        'h2d_mean'   : np.mean(_t['h2d']),
        'enc_mean'   : np.mean(_t['enc']),
        'nar_mean'   : np.mean(_t['nar']),
        'write_mean' : np.mean(_t['write']),
        'total_mean' : np.mean(_t['total']),
        'total_p50'  : _p(_t['total'], 50),
        'total_p95'  : _p(_t['total'], 95),
        'throughput' : np.mean(_t['tp']),
        'raw'        : _t,
    }
    s = timing_baseline[bs]
    print(f'  spectra={n_spec}  batches={len(_t["total"])}')
    print(f'  total : {s["total_mean"]:7.2f} ms/spec  p50={s["total_p50"]:.2f}  p95={s["total_p95"]:.2f}')
    print(f'  enc   : {s["enc_mean"]:7.2f} ms/spec  nar={s["nar_mean"]:.2f} ms/spec')
    print(f'  tp    : {s["throughput"]:.1f} spec/s')

    if DEVICE == 'cuda':
        torch.cuda.empty_cache()

_stop_gpu_bl.set()
time.sleep(1.0)

gpu_util_mean_bl = np.mean([s[0] for s in _gpu_samples_bl]) if _gpu_samples_bl else 0
gpu_vram_peak_bl = np.max([s[1] for s in _gpu_samples_bl]) if _gpu_samples_bl else 0

b1 = timing_baseline[1]
rb = b1['raw']
df_stage_baseline = pd.DataFrame([
    {'Stage': 'DataLoader fetch',    'mean_ms': b1['fetch_mean'],
     'p50_ms': np.percentile(rb['fetch'], 50), 'p95_ms': np.percentile(rb['fetch'], 95)},
    {'Stage': 'H2D transfer',        'mean_ms': b1['h2d_mean'],
     'p50_ms': np.percentile(rb['h2d'],   50), 'p95_ms': np.percentile(rb['h2d'],   95)},
    {'Stage': 'SpectrumEncoder',     'mean_ms': b1['enc_mean'],
     'p50_ms': np.percentile(rb['enc'],   50), 'p95_ms': np.percentile(rb['enc'],   95)},
    {'Stage': 'NAR Decoder (1 pass)','mean_ms': b1['nar_mean'],
     'p50_ms': np.percentile(rb['nar'],   50), 'p95_ms': np.percentile(rb['nar'],   95)},
    {'Stage': 'Output write',        'mean_ms': b1['write_mean'],
     'p50_ms': np.percentile(rb['write'], 50), 'p95_ms': np.percentile(rb['write'], 95)},
    {'Stage': 'TOTAL per spectrum',  'mean_ms': b1['total_mean'],
     'p50_ms': b1['total_p50'],                'p95_ms': b1['total_p95']},
]).round(3)

df_throughput_baseline = pd.DataFrame([{
    'batch_size'        : bs,
    'total_ms_per_spec' : timing_baseline[bs]['total_mean'],
    'throughput_spec_s' : timing_baseline[bs]['throughput'],
    'enc_ms_per_spec'   : timing_baseline[bs]['enc_mean'],
    'nar_ms_per_spec'   : timing_baseline[bs]['nar_mean'],
    'p50_ms'            : timing_baseline[bs]['total_p50'],
    'p95_ms'            : timing_baseline[bs]['total_p95'],
} for bs in BATCH_SIZES]).round(3)

print(f'\n── Stage breakdown (BASELINE FP32, bs=1, {b1["n_spec"]} spectra) ──')
print(df_stage_baseline.to_string(index=False))
print(f'\n── Multi-batch throughput (BASELINE FP32) ──')
print(df_throughput_baseline.to_string(index=False))
print(f'\nGPU util (mean): {gpu_util_mean_bl:.0f}%  |  Peak VRAM: {gpu_vram_peak_bl:.2f} GB')

_total_ms_bl = b1['total_mean']
_msg_35 = 'MEETS ✓' if _total_ms_bl <= 35 else f'FAILS — {_total_ms_bl:.1f} ms'
_msg_50 = 'MEETS ✓' if _total_ms_bl <= 50 else f'FAILS — {_total_ms_bl:.1f} ms'
_msg_10 = 'MEETS ✓' if _total_ms_bl <= 10 else f'FAILS — {_total_ms_bl:.1f} ms'
print(f'35 ms (~29 Hz) target (bs=1): {_msg_35}')
print(f'50 ms (20 Hz)  target (bs=1): {_msg_50}')
print(f'10 ms (100 Hz) target (bs=1): {_msg_10}')

df_stage_baseline.to_csv('results/nar_baseline_stage_timing_bs1.csv', index=False)
df_throughput_baseline.to_csv('results/nar_baseline_throughput_all_bs.csv', index=False)
print('\nSaved: nar_baseline_stage_timing_bs1.csv | nar_baseline_throughput_all_bs.csv')


══ Timing BASELINE NAR (FP32, eager)  batch_size=   1 ══


subset_profile.mgf: 0 spectra [00:00, ? spectra/s]

  bs=1: 100%|██████████| 5000/5000 [01:54<00:00, 43.84spec/s]


  spectra=5000  batches=5000
  total :   22.43 ms/spec  p50=21.51  p95=29.62
  enc   :    7.98 ms/spec  nar=12.78 ms/spec
  tp    : 45.1 spec/s

══ Timing BASELINE NAR (FP32, eager)  batch_size=   8 ══


subset_profile.mgf: 0 spectra [00:00, ? spectra/s]

...

  bs=128: 5120spec [00:09, 515.35spec/s]                        

  spectra=5120  batches=40
  total :    1.93 ms/spec  p50=1.91  p95=2.05
  enc   :    0.54 ms/spec  nar=1.18 ms/spec
  tp    : 518.7 spec/s

══ Timing BASELINE NAR (FP32, eager)  batch_size= 512 ══


subset_profile.mgf: 0 spectra [00:00, ? spectra/s]

  bs=512: 5120spec [00:10, 481.39spec/s]                        


  spectra=5120  batches=10
  total :    2.07 ms/spec  p50=2.08  p95=2.12
  enc   :    0.64 ms/spec  nar=1.26 ms/spec
  tp    : 482.2 spec/s

── Stage breakdown (BASELINE FP32, bs=1, 5000 spectra) ──
               Stage  mean_ms  p50_ms  p95_ms
    DataLoader fetch    1.356   1.298   1.833
        H2D transfer    0.214   0.200   0.299
     SpectrumEncoder    7.984   7.550  11.202
NAR Decoder (1 pass)   12.777  12.253  16.333
        Output write    0.102   0.095   0.132
  TOTAL per spectrum   22.432  21.511  29.617

── Multi-batch throughput (BASELINE FP32) ──
 batch_size  total_ms_per_spec  throughput_spec_s  enc_ms_per_spec  nar_ms_per_spec  p50_ms  p95_ms
          1             22.432             45.109            7.984           12.777  21.511  29.617
          8              3.228            315.950            1.077            1.741   3.004   4.342
         32              1.867            536.536            0.577            1.028   1.860   1.984
        128              1.929   

In [23]:
# ═══════════════════════════════════════════════════════════════════════
# CELL 7 (NEW) — BF16 Mixed-Precision NAR Timing
# Official inference pattern: torch.no_grad() + torch.autocast(device_type=
# 'cuda', dtype=torch.bfloat16) wrapping ONLY the model forward calls.
# Model weights remain FP32 — no model.to(bfloat16) anywhere.
# ═══════════════════════════════════════════════════════════════════════
assert BF16_SUPPORTED, 'BF16 not supported on this GPU — cannot run this cell.'

_gpu_samples_bf = []
_stop_gpu_bf    = threading.Event()

def _gpu_monitor_bf():
    import subprocess as _sp
    while not _stop_gpu_bf.is_set():
        r = _sp.run(['nvidia-smi',
                     '--query-gpu=utilization.gpu,memory.used',
                     '--format=csv,noheader,nounits'],
                    capture_output=True, text=True)
        if r.returncode == 0:
            try:
                u, m = r.stdout.strip().split(', ')
                _gpu_samples_bf.append((int(u), float(m) / 1024))
            except Exception:
                pass
        time.sleep(0.5)

threading.Thread(target=_gpu_monitor_bf, daemon=True).start()

timing_bf16 = {}

for bs in BATCH_SIZES:
    print(f'\n══ Timing NAR + BF16  batch_size={bs:4d} ══')

    _dm_bs = DeNovoDataModule(
        lance_dir=LANCE_DIR,
        test_paths=[SUBSET_MGF],
        eval_batch_size=bs,
        tokenizer=runner.model.tokenizer,
        max_charge=MODEL_MAX_CHARGE,
        n_workers=0,
    )
    _dm_bs.setup(stage='test', annotated=False)

    _w_iter = iter(_dm_bs.predict_dataloader())
    with torch.no_grad(), torch.autocast(device_type='cuda', dtype=BF16_DTYPE):
        for _w in range(N_WARMUP_BATCHES):
            try:
                _wb = next(_w_iter)
            except StopIteration:
                break
            _wm, _wi, _wp, _ = model._process_batch(_wb)
            _wm = _wm.to(DEVICE); _wi = _wi.to(DEVICE); _wp = _wp.to(DEVICE)
            _wme, _wmk = model.encoder(_wm, _wi)
            _wz = torch.zeros((_wm.shape[0], model.max_peptide_len),
                               dtype=torch.long, device=DEVICE)
            model.decoder(tokens=_wz, memory=_wme,
                          memory_key_padding_mask=_wmk, precursors=_wp)
    del _w_iter
    _sync()

    _t = {k: [] for k in ['fetch', 'h2d', 'enc', 'nar', 'write', 'total', 'tp']}
    _loader = _dm_bs.predict_dataloader()
    _it     = iter(_loader)
    n_spec  = 0
    pbar    = tqdm(total=N_TIMING_SPECTRA, desc=f'  bs={bs}', unit='spec')

    while n_spec < N_TIMING_SPECTRA:
        # Fetch / H2D stay OUTSIDE autocast — they're plain data movement,
        # not model compute (matches official guidance: autocast wraps
        # only the forward pass).
        _sync(); t0 = time.perf_counter()
        try:
            batch = next(_it)
        except StopIteration:
            _it = iter(_loader)
            batch = next(_it)
        t_fetch = (time.perf_counter() - t0) * 1000

        _sync(); t0 = time.perf_counter()
        mzs, ints, precs, _ = model._process_batch(batch)
        mzs   = mzs.to(DEVICE)
        ints  = ints.to(DEVICE)
        precs = precs.to(DEVICE)
        _sync(); t_h2d = (time.perf_counter() - t0) * 1000
        actual_bs = mzs.shape[0]

        with torch.no_grad(), torch.autocast(device_type='cuda', dtype=BF16_DTYPE):
            _sync(); t0 = time.perf_counter()
            memories, mem_masks = model.encoder(mzs, ints)
            _sync(); t_enc = (time.perf_counter() - t0) * 1000

            zero_tokens = torch.zeros(
                (actual_bs, model.max_peptide_len),
                dtype=torch.long, device=DEVICE)
            _sync(); t0 = time.perf_counter()
            scores = model.decoder(
                tokens=zero_tokens,
                memory=memories,
                memory_key_padding_mask=mem_masks,
                precursors=precs,
            )
            _sync(); t_nar = (time.perf_counter() - t0) * 1000

        # argmax() always returns int64 regardless of scores' dtype (bf16
        # here) — no explicit .float() cast needed before .cpu()/.tolist().
        t0 = time.perf_counter()
        pred_tok = scores.argmax(dim=-1).cpu()
        _out = [{'tokens': tok.tolist()} for tok in pred_tok]
        t_write = (time.perf_counter() - t0) * 1000

        t_total = t_fetch + t_h2d + t_enc + t_nar + t_write

        _t['fetch'].append(t_fetch  / actual_bs)
        _t['h2d'].append(t_h2d     / actual_bs)
        _t['enc'].append(t_enc     / actual_bs)
        _t['nar'].append(t_nar     / actual_bs)
        _t['write'].append(t_write / actual_bs)
        _t['total'].append(t_total / actual_bs)
        _t['tp'].append(actual_bs  / (t_total / 1000))

        n_spec += actual_bs
        pbar.update(actual_bs)
        if n_spec >= N_TIMING_SPECTRA:
            break

    pbar.close()

    def _p(a, q): return np.percentile(a, q)
    timing_bf16[bs] = {
        'n_spec'     : n_spec,
        'n_batches'  : len(_t['total']),
        'fetch_mean' : np.mean(_t['fetch']),
        'h2d_mean'   : np.mean(_t['h2d']),
        'enc_mean'   : np.mean(_t['enc']),
        'nar_mean'   : np.mean(_t['nar']),
        'write_mean' : np.mean(_t['write']),
        'total_mean' : np.mean(_t['total']),
        'total_p50'  : _p(_t['total'], 50),
        'total_p95'  : _p(_t['total'], 95),
        'throughput' : np.mean(_t['tp']),
        'raw'        : _t,
    }
    s = timing_bf16[bs]
    print(f'  spectra={n_spec}  batches={len(_t["total"])}')
    print(f'  total : {s["total_mean"]:7.2f} ms/spec  p50={s["total_p50"]:.2f}  p95={s["total_p95"]:.2f}')
    print(f'  enc   : {s["enc_mean"]:7.2f} ms/spec  nar={s["nar_mean"]:.2f} ms/spec')
    print(f'  tp    : {s["throughput"]:.1f} spec/s')

    if DEVICE == 'cuda':
        torch.cuda.empty_cache()

_stop_gpu_bf.set()
time.sleep(1.0)

gpu_util_mean_bf = np.mean([s[0] for s in _gpu_samples_bf]) if _gpu_samples_bf else 0
gpu_vram_peak_bf = np.max([s[1] for s in _gpu_samples_bf]) if _gpu_samples_bf else 0

c1 = timing_bf16[1]
rc = c1['raw']
df_stage_bf16 = pd.DataFrame([
    {'Stage': 'DataLoader fetch',          'mean_ms': c1['fetch_mean'],
     'p50_ms': np.percentile(rc['fetch'], 50), 'p95_ms': np.percentile(rc['fetch'], 95)},
    {'Stage': 'H2D transfer',              'mean_ms': c1['h2d_mean'],
     'p50_ms': np.percentile(rc['h2d'],   50), 'p95_ms': np.percentile(rc['h2d'],   95)},
    {'Stage': 'SpectrumEncoder (BF16)',    'mean_ms': c1['enc_mean'],
     'p50_ms': np.percentile(rc['enc'],   50), 'p95_ms': np.percentile(rc['enc'],   95)},
    {'Stage': 'NAR Decoder (BF16, 1 pass)','mean_ms': c1['nar_mean'],
     'p50_ms': np.percentile(rc['nar'],   50), 'p95_ms': np.percentile(rc['nar'],   95)},
    {'Stage': 'Output write',              'mean_ms': c1['write_mean'],
     'p50_ms': np.percentile(rc['write'], 50), 'p95_ms': np.percentile(rc['write'], 95)},
    {'Stage': 'TOTAL per spectrum',        'mean_ms': c1['total_mean'],
     'p50_ms': c1['total_p50'],                'p95_ms': c1['total_p95']},
]).round(3)

df_throughput_bf16 = pd.DataFrame([{
    'batch_size'        : bs,
    'total_ms_per_spec' : timing_bf16[bs]['total_mean'],
    'throughput_spec_s' : timing_bf16[bs]['throughput'],
    'enc_ms_per_spec'   : timing_bf16[bs]['enc_mean'],
    'nar_ms_per_spec'   : timing_bf16[bs]['nar_mean'],
    'p50_ms'            : timing_bf16[bs]['total_p50'],
    'p95_ms'            : timing_bf16[bs]['total_p95'],
} for bs in BATCH_SIZES]).round(3)

print(f'\n── Stage breakdown (BF16, bs=1, {c1["n_spec"]} spectra) ──')
print(df_stage_bf16.to_string(index=False))
print(f'\n── Multi-batch throughput (BF16) ──')
print(df_throughput_bf16.to_string(index=False))
print(f'\nGPU util (mean): {gpu_util_mean_bf:.0f}%  |  Peak VRAM: {gpu_vram_peak_bf:.2f} GB')

_total_ms_bf = c1['total_mean']
_msg_35c = 'MEETS ✓' if _total_ms_bf <= 35 else f'FAILS — {_total_ms_bf:.1f} ms'
_msg_50c = 'MEETS ✓' if _total_ms_bf <= 50 else f'FAILS — {_total_ms_bf:.1f} ms'
_msg_10c = 'MEETS ✓' if _total_ms_bf <= 10 else f'FAILS — {_total_ms_bf:.1f} ms'
print(f'35 ms (~29 Hz) target (bs=1): {_msg_35c}')
print(f'50 ms (20 Hz)  target (bs=1): {_msg_50c}')
print(f'10 ms (100 Hz) target (bs=1): {_msg_10c}')

_speedup_bs1 = timing_baseline[1]['total_mean'] / max(c1['total_mean'], 0.001)
print(f'\nSpeedup vs baseline FP32 (bs=1): {_speedup_bs1:.2f}×')

df_stage_bf16.to_csv('results/nar_bf16_stage_timing_bs1.csv', index=False)
df_throughput_bf16.to_csv('results/nar_bf16_throughput_all_bs.csv', index=False)
print('\nSaved: nar_bf16_stage_timing_bs1.csv | nar_bf16_throughput_all_bs.csv')


══ Timing NAR + BF16  batch_size=   1 ══


subset_profile.mgf: 0 spectra [00:00, ? spectra/s]

  bs=1: 100%|██████████| 5000/5000 [02:19<00:00, 35.73spec/s]


  spectra=5000  batches=5000
  total :   27.34 ms/spec  p50=26.32  p95=35.35
  enc   :    9.31 ms/spec  nar=16.32 ms/spec
  tp    : 36.9 spec/s

══ Timing NAR + BF16  batch_size=   8 ══


subset_profile.mgf: 0 spectra [00:00, ? spectra/s]

...

  bs=128: 5120spec [00:05, 911.07spec/s]                        


  spectra=5120  batches=40
  total :    1.08 ms/spec  p50=1.06  p95=1.18
  enc   :    0.37 ms/spec  nar=0.51 ms/spec
  tp    : 923.9 spec/s

══ Timing NAR + BF16  batch_size= 512 ══


subset_profile.mgf: 0 spectra [00:00, ? spectra/s]

  bs=512: 5120spec [00:05, 886.18spec/s]                        


  spectra=5120  batches=10
  total :    1.12 ms/spec  p50=1.13  p95=1.16
  enc   :    0.40 ms/spec  nar=0.51 ms/spec
  tp    : 890.2 spec/s

── Stage breakdown (BF16, bs=1, 5000 spectra) ──
                     Stage  mean_ms  p50_ms  p95_ms
          DataLoader fetch    1.377   1.327   1.865
              H2D transfer    0.219   0.206   0.301
    SpectrumEncoder (BF16)    9.307   8.826  13.065
NAR Decoder (BF16, 1 pass)   16.320  15.726  20.154
              Output write    0.120   0.113   0.156
        TOTAL per spectrum   27.344  26.321  35.346

── Multi-batch throughput (BF16) ──
 batch_size  total_ms_per_spec  throughput_spec_s  enc_ms_per_spec  nar_ms_per_spec  p50_ms  p95_ms
          1             27.344             36.945            9.307           16.320  26.321  35.346
          8              3.939            257.555            1.269            2.260   3.722   5.175
         32              1.184            859.638            0.338            0.593   1.120   1.551
        1

In [24]:
# ═══════════════════════════════════════════════════════════════════════
# CELL 8 — torch.profiler: Baseline FP32 vs BF16 (bs=1, ≥50 spectra)
# Includes an explicit, evidence-based check of which SDPA attention
# kernel actually ran — answers "does BF16 auto-enable FlashAttention?"
# from the real kernel trace rather than assumption.
# ═══════════════════════════════════════════════════════════════════════
ACTS           = ([ProfilerActivity.CPU, ProfilerActivity.CUDA]
                  if DEVICE == 'cuda' else [ProfilerActivity.CPU])
SORT_KEY       = 'cpu_time_total'
N_PROF_BATCHES = PROF_WARMUP + PROF_ACTIVE

print(f'Pre-fetching {N_PROF_BATCHES} bs=1 batches for profiler…')
_dm_prof = DeNovoDataModule(
    lance_dir=LANCE_DIR,
    test_paths=[SUBSET_MGF],
    eval_batch_size=1,
    tokenizer=runner.model.tokenizer,
    max_charge=MODEL_MAX_CHARGE,
    n_workers=0,
)
_dm_prof.setup(stage='test', annotated=False)

_prof_batches, _skipped = [], 0
for _b in _dm_prof.predict_dataloader():
    _mz, _it, _pr, _ = model._process_batch(_b)
    if _pr[0, 1].item() > MODEL_MAX_CHARGE:
        _skipped += 1; continue
    _prof_batches.append((_mz.to(DEVICE), _it.to(DEVICE), _pr.to(DEVICE)))
    if len(_prof_batches) >= N_PROF_BATCHES:
        break

if _skipped:
    print(f'  Skipped {_skipped} out-of-range-charge spectra')
if len(_prof_batches) == 0:
    raise RuntimeError('No valid batches for profiling.')
while len(_prof_batches) < N_PROF_BATCHES:
    _prof_batches.extend(_prof_batches[:N_PROF_BATCHES - len(_prof_batches)])
print(f'Using {len(_prof_batches)} batches')

_zero_toks = torch.zeros((1, model.max_peptide_len), dtype=torch.long, device=DEVICE)

def _run_profiler(label, trace_path, txt_path, use_bf16, n_warm=10):
    def _ctx():
        if use_bf16:
            return torch.autocast(device_type='cuda', dtype=BF16_DTYPE)
        return torch.autocast(device_type='cuda', dtype=BF16_DTYPE, enabled=False)

    with torch.no_grad(), _ctx():
        for _mz, _it, _pr in _prof_batches[:n_warm]:
            _me, _mk = model.encoder(_mz, _it)
            model.decoder(tokens=_zero_toks, memory=_me,
                          memory_key_padding_mask=_mk, precursors=_pr)
    _sync()

    _store = {}
    def _on_ready(p):
        p.export_chrome_trace(trace_path)
        _store['tbl']  = p.key_averages().table(sort_by=SORT_KEY, row_limit=12)
        _store['avgs'] = p.key_averages()

    with profile(
        activities=ACTS,
        record_shapes=True,
        schedule=schedule(wait=0, warmup=PROF_WARMUP, active=PROF_ACTIVE),
        on_trace_ready=_on_ready,
    ) as p:
        with torch.no_grad(), _ctx():
            for _mz, _it, _pr in _prof_batches:
                with record_function(label):
                    _me, _mk = model.encoder(_mz, _it)
                    model.decoder(tokens=_zero_toks, memory=_me,
                                  memory_key_padding_mask=_mk, precursors=_pr)
                _sync()
                p.step()

    print(_store.get('tbl', '(no profiler data)'))
    with open(txt_path, 'w') as fh:
        fh.write(f'{label}  bs=1  warmup={PROF_WARMUP}  active={PROF_ACTIVE}\n')
        fh.write('=' * 64 + '\n')
        fh.write(str(_store.get('tbl', 'no data')))
    print(f'Chrome trace → {trace_path}')
    if DEVICE == 'cuda':
        torch.cuda.synchronize(); torch.cuda.empty_cache()
    return _store

def _detect_attention_kernel(store, label):
    """Inspects the captured kernel-average entries for the exact SDPA
    backend that ran — definitive evidence, not inference from dtype."""
    if not (DEVICE == 'cuda' and store.get('avgs')):
        print(f'{label}: no CUDA profiler data available')
        return
    flash = next((e for e in store['avgs'] if 'flash_attention' in e.key), None)
    eff   = next((e for e in store['avgs'] if 'efficient_attention' in e.key), None)
    if flash is not None:
        print(f'{label}: FlashAttention kernel ACTIVE  '
              f'({flash.key}, {flash.count} calls)')
    elif eff is not None:
        print(f'{label}: memory-EFFICIENT attention active (NOT Flash)  '
              f'({eff.key}, {eff.count} calls)')
    else:
        print(f'{label}: neither flash nor efficient attention kernel found in trace')

# ════════════════════════════════════════════════════════════════════
# A) BASELINE FP32 — SpectrumEncoder only
# ════════════════════════════════════════════════════════════════════
print('\n── A) torch.profiler: BASELINE FP32 SpectrumEncoder (bs=1) ──')
_store_bl_enc = {}
def _on_ready_bl_enc(p):
    p.export_chrome_trace('results/trace_baseline_encoder.json')
    _store_bl_enc['tbl']  = p.key_averages().table(sort_by=SORT_KEY, row_limit=10)
    _store_bl_enc['avgs'] = p.key_averages()
with torch.no_grad():
    for _mz, _it, _pr in _prof_batches[:10]:
        model.encoder(_mz, _it)
_sync()
with profile(activities=ACTS, record_shapes=True,
             schedule=schedule(wait=0, warmup=PROF_WARMUP, active=PROF_ACTIVE),
             on_trace_ready=_on_ready_bl_enc) as p_bl_enc:
    with torch.no_grad():
        for _mz, _it, _pr in _prof_batches:
            with record_function('baseline_encoder'):
                model.encoder(_mz, _it)
            _sync(); p_bl_enc.step()
print(_store_bl_enc.get('tbl', '(no data)'))
with open('results/profiler_baseline_encoder.txt', 'w') as fh:
    fh.write(f'BASELINE FP32 SpectrumEncoder  bs=1  warmup={PROF_WARMUP}  active={PROF_ACTIVE}\n')
    fh.write('=' * 64 + '\n')
    fh.write(str(_store_bl_enc.get('tbl', 'no data')))
print('Chrome trace → results/trace_baseline_encoder.json')
if DEVICE == 'cuda': torch.cuda.synchronize(); torch.cuda.empty_cache()

# ════════════════════════════════════════════════════════════════════
# B) BASELINE FP32 — Full NAR forward
# ════════════════════════════════════════════════════════════════════
print('\n── B) torch.profiler: BASELINE FP32 Full NAR forward (bs=1) ──')
_store_bl_full = _run_profiler(
    'baseline_full_forward', 'results/trace_baseline_full.json',
    'results/profiler_baseline_full.txt', use_bf16=False)
_detect_attention_kernel(_store_bl_full, 'Baseline FP32 (full forward)')

# ════════════════════════════════════════════════════════════════════
# C) BF16 — SpectrumEncoder only
# ════════════════════════════════════════════════════════════════════
print('\n── C) torch.profiler: BF16 SpectrumEncoder (bs=1) ──')
_store_bf_enc = {}
def _on_ready_bf_enc(p):
    p.export_chrome_trace('results/trace_bf16_encoder.json')
    _store_bf_enc['tbl']  = p.key_averages().table(sort_by=SORT_KEY, row_limit=10)
    _store_bf_enc['avgs'] = p.key_averages()
with torch.no_grad(), torch.autocast(device_type='cuda', dtype=BF16_DTYPE):
    for _mz, _it, _pr in _prof_batches[:10]:
        model.encoder(_mz, _it)
_sync()
with profile(activities=ACTS, record_shapes=True,
             schedule=schedule(wait=0, warmup=PROF_WARMUP, active=PROF_ACTIVE),
             on_trace_ready=_on_ready_bf_enc) as p_bf_enc:
    with torch.no_grad(), torch.autocast(device_type='cuda', dtype=BF16_DTYPE):
        for _mz, _it, _pr in _prof_batches:
            with record_function('bf16_encoder'):
                model.encoder(_mz, _it)
            _sync(); p_bf_enc.step()
print(_store_bf_enc.get('tbl', '(no data)'))
with open('results/profiler_bf16_encoder.txt', 'w') as fh:
    fh.write(f'BF16 SpectrumEncoder  bs=1  warmup={PROF_WARMUP}  active={PROF_ACTIVE}\n')
    fh.write('=' * 64 + '\n')
    fh.write(str(_store_bf_enc.get('tbl', 'no data')))
print('Chrome trace → results/trace_bf16_encoder.json')
if DEVICE == 'cuda': torch.cuda.synchronize(); torch.cuda.empty_cache()

# ════════════════════════════════════════════════════════════════════
# D) BF16 — Full NAR forward
# ════════════════════════════════════════════════════════════════════
print('\n── D) torch.profiler: BF16 Full NAR forward (bs=1) ──')
_store_bf_full = _run_profiler(
    'bf16_full_forward', 'results/trace_bf16_full.json',
    'results/profiler_bf16_full.txt', use_bf16=True)
_detect_attention_kernel(_store_bf_full, 'BF16 (full forward)')

# ── Kernel launch comparison ───────────────────────────────────────────
def _launch_count(store):
    if not (DEVICE == 'cuda' and store.get('avgs')):
        return None
    e = next((e for e in store['avgs'] if e.key == 'cudaLaunchKernel'), None)
    return e.count if e is not None else None

_launches_bl = _launch_count(_store_bl_full)
_launches_bf = _launch_count(_store_bf_full)

print('\n── Kernel launch comparison (Full NAR forward, bs=1, 50 profiled spectra) ──')
if _launches_bl is not None:
    print(f'Baseline FP32 : {_launches_bl/PROF_ACTIVE:.0f} launches/spectrum')
if _launches_bf is not None:
    print(f'BF16          : {_launches_bf/PROF_ACTIVE:.0f} launches/spectrum')

Pre-fetching 70 bs=1 batches for profiler…


subset_profile.mgf: 0 spectra [00:00, ? spectra/s]

Using 70 batches

── A) torch.profiler: BASELINE FP32 SpectrumEncoder (bs=1) ──
-------------------------------------------------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  
                                                   Name    Self CPU %      Self CPU   CPU total %     CPU total  CPU time avg     Self CUDA   Self CUDA %    CUDA total  CUDA time avg    # of Calls  
-------------------------------------------------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  
                                          ProfilerStep*         0.56%       4.283ms       100.00%     761.414ms      15.228ms       0.000us         0.00%      56.796ms       1.136ms            50  
                                       baseline_encoder        13.23%     100.765ms        99.39%     756.765ms      15.135ms  

In [25]:
# ═══════════════════════════════════════════════════════════════════════
# CELL 9 — Synthetic Micro-benchmark: Baseline FP32 vs BF16 (bs=1, 20 reps)
# ═══════════════════════════════════════════════════════════════════════
def make_synth_batch(bs=1, device=DEVICE):
    n_real = 123
    mzs_s  = torch.zeros(bs, N_PEAKS, device=device)
    ints_s = torch.zeros(bs, N_PEAKS, device=device)
    for i in range(bs):
        mzs_s[i, :n_real]  = torch.rand(n_real, device=device) * 1303 + 301
        ints_s[i, :n_real] = torch.rand(n_real, device=device)
        norm = ints_s[i, :n_real].norm().clamp(min=1e-8)
        ints_s[i, :n_real] /= norm
    charge = 2.0; pmz = 600.0
    precs  = torch.tensor(
        [[(pmz - 1.007276) * charge, charge, pmz]] * bs,
        dtype=torch.float, device=device)
    return mzs_s, ints_s, precs

smzs, sints, sprecs = make_synth_batch(bs=1)
szero = torch.zeros((1, model.max_peptide_len), dtype=torch.long, device=DEVICE)

def _time_pair(use_bf16, n_reps=20, n_warm=5):
    def _ctx():
        if use_bf16:
            return torch.autocast(device_type='cuda', dtype=BF16_DTYPE)
        return torch.autocast(device_type='cuda', dtype=BF16_DTYPE, enabled=False)

    with torch.no_grad(), _ctx():
        for _ in range(n_warm):
            _me, _mk = model.encoder(smzs, sints)
            model.decoder(tokens=szero, memory=_me,
                          memory_key_padding_mask=_mk, precursors=sprecs)
    _sync()
    with torch.no_grad(), _ctx():
        _me, _mk = model.encoder(smzs, sints)
    _sync()

    _et, _dt, _ft = [], [], []
    with torch.no_grad(), _ctx():
        for _ in range(n_reps):
            _sync(); t0 = time.perf_counter()
            model.encoder(smzs, sints)
            _sync(); _et.append((time.perf_counter() - t0) * 1000)
        for _ in range(n_reps):
            _sync(); t0 = time.perf_counter()
            model.decoder(tokens=szero, memory=_me,
                          memory_key_padding_mask=_mk, precursors=sprecs)
            _sync(); _dt.append((time.perf_counter() - t0) * 1000)
        for _ in range(n_reps):
            _sync(); t0 = time.perf_counter()
            _me2, _mk2 = model.encoder(smzs, sints)
            model.decoder(tokens=szero, memory=_me2,
                          memory_key_padding_mask=_mk2, precursors=sprecs)
            _sync(); _ft.append((time.perf_counter() - t0) * 1000)

    return (float(np.mean(_et[5:])), float(np.mean(_dt[5:])), float(np.mean(_ft[5:])))

baseline_enc_ms, baseline_dec_ms, baseline_full_ms = _time_pair(use_bf16=False)
bf16_enc_ms, bf16_dec_ms, bf16_full_ms             = _time_pair(use_bf16=True)

baseline_synth_tp = 1000.0 / baseline_full_ms
bf16_synth_tp     = 1000.0 / bf16_full_ms

print(f'\n── Synthetic Micro-timings (20 reps, drop first 5, bs=1) ──')
print(f'{"Metric":<30} {"Baseline":>12} {"BF16":>12} {"Speedup":>9}')
print('-' * 65)
_rows = [
    ('SpectrumEncoder (ms)', baseline_enc_ms,  bf16_enc_ms),
    ('NAR Decoder (ms)',     baseline_dec_ms,  bf16_dec_ms),
    ('Full forward (ms)',    baseline_full_ms, bf16_full_ms),
    ('Throughput (spec/s)',  baseline_synth_tp, bf16_synth_tp),
]
for label, bv, cv in _rows:
    spd = (bv / max(cv, 0.001)) if 'Throughput' not in label else (cv / max(bv, 0.001))
    print(f'  {label:<28} {bv:>12.2f} {cv:>12.2f} {spd:>8.2f}×')

pd.DataFrame({
    'metric'  : ['enc_ms', 'dec_ms', 'full_ms', 'throughput_spec_s'],
    'baseline': [baseline_enc_ms, baseline_dec_ms, baseline_full_ms, baseline_synth_tp],
    'bf16'    : [bf16_enc_ms, bf16_dec_ms, bf16_full_ms, bf16_synth_tp],
    'speedup' : [baseline_enc_ms/max(bf16_enc_ms,0.001),
                 baseline_dec_ms/max(bf16_dec_ms,0.001),
                 baseline_full_ms/max(bf16_full_ms,0.001),
                 bf16_synth_tp/max(baseline_synth_tp,0.001)],
}).to_csv('results/nar_baseline_vs_bf16_synthetic.csv', index=False)
print('\nSaved: results/nar_baseline_vs_bf16_synthetic.csv')


── Synthetic Micro-timings (20 reps, drop first 5, bs=1) ──
Metric                             Baseline         BF16   Speedup
-----------------------------------------------------------------
  SpectrumEncoder (ms)                 9.00         9.31     0.97×
  NAR Decoder (ms)                    15.54        17.73     0.88×
  Full forward (ms)                   24.79        27.63     0.90×
  Throughput (spec/s)                 40.33        36.19     0.90×

Saved: results/nar_baseline_vs_bf16_synthetic.csv


In [26]:
# ═══════════════════════════════════════════════════════════════════════
# CELL 10 — Plots + Summary: Baseline FP32 vs BF16
# ═══════════════════════════════════════════════════════════════════════
import datetime

b1 = timing_baseline[1]
c1 = timing_bf16[1]

_lats_bl = [timing_baseline[bs]['total_mean'] for bs in BATCH_SIZES]
_lats_bf = [timing_bf16[bs]['total_mean'] for bs in BATCH_SIZES]
_tps_bl  = [timing_baseline[bs]['throughput'] for bs in BATCH_SIZES]
_tps_bf  = [timing_bf16[bs]['throughput'] for bs in BATCH_SIZES]
_xi   = list(range(len(BATCH_SIZES)))
_xlbl = [str(b) for b in BATCH_SIZES]

# ── Figure 1 — Stage breakdown ────────────────────────────────────────
fig1, ax1 = plt.subplots(1, 2, figsize=(14, 5))
fig1.suptitle('Stage Breakdown (bs=1) — Baseline FP32 vs BF16', fontweight='bold')

_stages  = ['Fetch', 'H2D', 'Encoder', 'Decoder', 'Write']
_bl_vals = [b1['fetch_mean'], b1['h2d_mean'], b1['enc_mean'], b1['nar_mean'], b1['write_mean']]
_bf_vals = [c1['fetch_mean'], c1['h2d_mean'], c1['enc_mean'], c1['nar_mean'], c1['write_mean']]
_ymax = max(max(_bl_vals), max(_bf_vals)) * 1.25

for ax, vals, color, title in [
    (ax1[0], _bl_vals, '#D85A30', f'Baseline FP32\nTotal: {b1["total_mean"]:.1f} ms/spec'),
    (ax1[1], _bf_vals, '#1D9E75', f'BF16\nTotal: {c1["total_mean"]:.1f} ms/spec'),
]:
    bars = ax.bar(_stages, vals, color=color, edgecolor='none', width=0.55)
    for b, v in zip(bars, vals):
        ax.text(b.get_x() + b.get_width() / 2, b.get_height() + _ymax * 0.02, f'{v:.2f}', ha='center', fontsize=9)
    ax.set_title(title); ax.set_ylabel('ms / spectrum')
    ax.set_ylim(0, _ymax)
    ax.spines[['top', 'right']].set_visible(False)

plt.tight_layout()
plt.savefig('results/nar_stage_baseline_vs_bf16.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: results/nar_stage_baseline_vs_bf16.png')

# ── Figure 2 — Throughput & Latency vs Batch Size ─────────────────────
fig2, ax2 = plt.subplots(1, 2, figsize=(14, 5))
fig2.suptitle('NAR Performance vs Batch Size — Baseline FP32 vs BF16', fontweight='bold')

ax2[0].plot(_xi, _tps_bl, 'o-', color='#D85A30', lw=2, ms=8, label='Baseline FP32')
ax2[0].plot(_xi, _tps_bf, 's-', color='#1D9E75', lw=2, ms=8, label='BF16')
for i, (yb, yc) in enumerate(zip(_tps_bl, _tps_bf)):
    ax2[0].text(i, yb + max(_tps_bf) * 0.02, f'{yb:.0f}', ha='center', fontsize=8, color='#D85A30')
    ax2[0].text(i, yc + max(_tps_bf) * 0.06, f'{yc:.0f}', ha='center', fontsize=8, color='#1D9E75')
ax2[0].set_xticks(_xi); ax2[0].set_xticklabels(_xlbl)
ax2[0].set_xlabel('Batch size'); ax2[0].set_ylabel('Throughput (spec/s)')
ax2[0].set_title('Throughput vs Batch Size')
ax2[0].legend(fontsize=8, frameon=False)
ax2[0].spines[['top', 'right']].set_visible(False)

ax2[1].plot(_xi, _lats_bl, 'o-', color='#D85A30', lw=2, ms=8, label='Baseline FP32')
ax2[1].plot(_xi, _lats_bf, 's-', color='#1D9E75', lw=2, ms=8, label='BF16')
ax2[1].axhline(35, color='#9B59B6', lw=1.5, ls='--', label='35 ms (~29 Hz)')
ax2[1].axhline(50, color='#E67E22', lw=1.2, ls='-.', label='50 ms (20 Hz)')
ax2[1].axhline(10, color='black',   lw=1.5, ls=':',  label='10 ms (100 Hz target)')
for i, (yb, yc) in enumerate(zip(_lats_bl, _lats_bf)):
    ax2[1].text(i, yb + max(_lats_bl) * 0.02, f'{yb:.1f}', ha='center', fontsize=8, color='#D85A30')
    ax2[1].text(i, yc + max(_lats_bl) * 0.06, f'{yc:.1f}', ha='center', fontsize=8, color='#1D9E75')
ax2[1].set_xticks(_xi); ax2[1].set_xticklabels(_xlbl)
ax2[1].set_xlabel('Batch size'); ax2[1].set_ylabel('ms / spectrum')
ax2[1].set_title('Latency per Spectrum vs Batch Size')
ax2[1].legend(fontsize=8, frameon=False)
ax2[1].spines[['top', 'right']].set_visible(False)

plt.tight_layout()
plt.savefig('results/nar_performance_baseline_vs_bf16.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: results/nar_performance_baseline_vs_bf16.png')

# ── Figure 3 — Synthetic comparison + latency distribution overlay ────
fig3, ax3 = plt.subplots(1, 2, figsize=(14, 5))
fig3.suptitle('Synthetic Timing + Real Latency Distribution — Baseline vs BF16', fontweight='bold')

_spd_synth = baseline_full_ms / max(bf16_full_ms, 0.001)
bars3 = ax3[0].bar(['Baseline\nFP32', 'BF16'],
                    [baseline_full_ms, bf16_full_ms],
                    color=['#D85A30', '#1D9E75'], edgecolor='none', width=0.35)
for b, v, tp in zip(bars3, [baseline_full_ms, bf16_full_ms], [baseline_synth_tp, bf16_synth_tp]):
    ax3[0].text(b.get_x() + b.get_width() / 2, b.get_height() + max(baseline_full_ms, bf16_full_ms) * 0.02,
                f'{v:.1f} ms\n{tp:.1f} spec/s', ha='center', fontsize=10)
ax3[0].axhline(10, color='black', lw=1.5, ls=':', label='10 ms (100 Hz target)')
ax3[0].set_title(f'Full Forward Pass (synthetic, bs=1)\nSpeedup: {_spd_synth:.2f}×')
ax3[0].set_ylabel('ms / spectrum')
ax3[0].legend(fontsize=9, frameon=False)
ax3[0].spines[['top', 'right']].set_visible(False)

_raw_bl = timing_baseline[1]['raw']['total']
_raw_bf = timing_bf16[1]['raw']['total']
ax3[1].hist(_raw_bl, bins=25, color='#D85A30', alpha=0.55, label=f'Baseline (mean {np.mean(_raw_bl):.1f}ms)')
ax3[1].hist(_raw_bf, bins=25, color='#1D9E75', alpha=0.55, label=f'BF16 (mean {np.mean(_raw_bf):.1f}ms)')
ax3[1].axvline(10, color='black', lw=1.5, ls=':', alpha=0.6, label='10 ms (100 Hz target)')
ax3[1].legend(fontsize=8, frameon=False)
ax3[1].set_title(f'Real-Spectra Latency Distribution (bs=1, {len(_raw_bl)} spectra)')
ax3[1].set_xlabel('ms / spectrum')
ax3[1].spines[['top', 'right']].set_visible(False)

plt.tight_layout()
plt.savefig('results/nar_baseline_vs_bf16_comparison.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: results/nar_baseline_vs_bf16_comparison.png')

# ── Text Summary ────────────────────────────────────────────────────
_now = datetime.datetime.now().strftime('%Y-%m-%d %H:%M')

_total_bl = b1['total_mean']; _total_bf = c1['total_mean']
_t10_bl = 'MEETS ✓' if _total_bl <= 10 else f'FAILS ({_total_bl:.1f} ms, {_total_bl/10:.1f}× over)'
_t10_bf = 'MEETS ✓' if _total_bf <= 10 else f'FAILS ({_total_bf:.1f} ms, {_total_bf/10:.1f}× over)'
_speedup_real_bs1 = _total_bl / max(_total_bf, 0.001)

summary = f"""CASANOVO NAR PROFILING — Baseline (FP32) vs BF16 Mixed Precision
Generated  : {_now}
Hardware   : {GPU_NAME} | {TOTAL_VRAM:.1f} GB VRAM | PyTorch {torch.__version__}
Dataset    : {SUBSET_MGF} ({N_SUBSET} spectra) | Timed: {N_TIMING_SPECTRA} spectra per batch size
Batch sizes: {BATCH_SIZES}
BF16 supported: {BF16_SUPPORTED}

OPTIMIZATION APPLIED
  torch.no_grad() + torch.autocast(device_type='cuda', dtype=torch.bfloat16)
  wrapping ONLY the encoder/decoder forward calls. Model weights remain FP32.

REAL-SPECTRA STAGE BREAKDOWN (bs=1, {b1['n_spec']} spectra)
Baseline FP32:
{df_stage_baseline.to_string(index=False)}
  Throughput : {b1['throughput']:.1f} spec/s | 10ms target: {_t10_bl}

BF16:
{df_stage_bf16.to_string(index=False)}
  Throughput : {c1['throughput']:.1f} spec/s | 10ms target: {_t10_bf}

Speedup (bs=1, real spectra): {_speedup_real_bs1:.2f}×

MULTI-BATCH THROUGHPUT
Baseline FP32:
{df_throughput_baseline.to_string(index=False)}

BF16:
{df_throughput_bf16.to_string(index=False)}

SYNTHETIC MICRO-BENCHMARK (20 reps, bs=1)
  Encoder — baseline: {baseline_enc_ms:.2f} ms | BF16: {bf16_enc_ms:.2f} ms
  Decoder — baseline: {baseline_dec_ms:.2f} ms | BF16: {bf16_dec_ms:.2f} ms
  Full    — baseline: {baseline_full_ms:.2f} ms ({baseline_synth_tp:.1f} spec/s) | BF16: {bf16_full_ms:.2f} ms ({bf16_synth_tp:.1f} spec/s)
  Speedup (synthetic): {_spd_synth:.2f}×

GPU UTILIZATION
  Baseline FP32 : {gpu_util_mean_bl:.0f}% mean util | {gpu_vram_peak_bl:.2f} GB peak VRAM
  BF16          : {gpu_util_mean_bf:.0f}% mean util | {gpu_vram_peak_bf:.2f} GB peak VRAM

ARTIFACTS SAVED TO results/
  nar_stage_baseline_vs_bf16.png
  nar_performance_baseline_vs_bf16.png
  nar_baseline_vs_bf16_comparison.png
  trace_baseline_encoder.json | trace_baseline_full.json
  trace_bf16_encoder.json     | trace_bf16_full.json   (ui.perfetto.dev)
  profiler_baseline_encoder.txt | profiler_baseline_full.txt
  profiler_bf16_encoder.txt     | profiler_bf16_full.txt
  nar_baseline_stage_timing_bs1.csv | nar_baseline_throughput_all_bs.csv
  nar_bf16_stage_timing_bs1.csv     | nar_bf16_throughput_all_bs.csv
  nar_baseline_vs_bf16_synthetic.csv
  nar_summary.txt   (this file)
"""

print(summary)
with open('results/nar_summary.txt', 'w') as fh:
    fh.write(summary)

print('\n── results/ ──')
for _f in sorted(os.listdir('results')):
    _fp = os.path.join('results', _f)
    print(f'  {_f:<55} {os.path.getsize(_fp)/1024:.1f} KB')
print('\nProfiling complete. Primary: results/nar_summary.txt')

Saved: results/nar_stage_baseline_vs_bf16.png
Saved: results/nar_performance_baseline_vs_bf16.png
Saved: results/nar_baseline_vs_bf16_comparison.png
CASANOVO NAR PROFILING — Baseline (FP32) vs BF16 Mixed Precision
Generated  : 2026-06-21 15:37
Hardware   : NVIDIA L4 | 23.6 GB VRAM | PyTorch 2.7.1+cu128
Dataset    : subset_profile.mgf (6000 spectra) | Timed: 5000 spectra per batch size
Batch sizes: [1, 8, 32, 128, 512]
BF16 supported: True

OPTIMIZATION APPLIED
  torch.no_grad() + torch.autocast(device_type='cuda', dtype=torch.bfloat16)
  wrapping ONLY the encoder/decoder forward calls. Model weights remain FP32.

REAL-SPECTRA STAGE BREAKDOWN (bs=1, 5000 spectra)
Baseline FP32:
               Stage  mean_ms  p50_ms  p95_ms
    DataLoader fetch    1.356   1.298   1.833
        H2D transfer    0.214   0.200   0.299
     SpectrumEncoder    7.984   7.550  11.202
NAR Decoder (1 pass)   12.777  12.253  16.333
        Output write    0.102   0.095   0.132
  TOTAL per spectrum   22.432  21.511 